# HS850811 市场模型复算底稿

## tl;dr

原五指标战略 TOPSIS 的前三为韩国、澳大利亚、马来西亚；日本排第 5。正文与图表未同步。

本笔记本是审计入口，全部结果来自本目录的当前数据。

## Context & Methods

保留战略权重 0.30/0.10/0.10/0.30/0.20、金额 log1p、Min-Max、TOPSIS，以及原随机种子与 5,000 次 Dirichlet 情景。

### Key Assumptions

从 ITC-analysis-result 目录运行；原始数据在 ../ITC-data。HS850811 不是扫地机器人专属。空值不填零，EPM 与贸易期不同。完整过程见 methodology.md。

## Data

读取共同完整样本，保留三个贸易覆盖缺失市场。来源包括 source_ledger_master.csv 与 model/input/input_lineage.csv。

In [1]:
from pathlib import Path
import csv, json, importlib.util, hashlib
import numpy as np
root = Path.cwd()
assert (root / 'model_inputs.csv').is_file(), 'Run from ITC-analysis-result'
spec = importlib.util.spec_from_file_location('itc_model', root / 'rebuild_itc_model.py')
itc = importlib.util.module_from_spec(spec)
spec.loader.exec_module(itc)
inputs = itc.read_csv(root / 'model_inputs.csv')
complete = []
for r in inputs:
    if all(r[k] != '' for k in ['imports_2025_usd_thousand','import_cagr_2020_2025_pct','china_share_2025_pct','epi_potential_usd','untapped_potential_usd']):
        complete.append({k: itc.number(v) if k not in ('country_code','country_name','source_ids') else v for k,v in r.items()})
print(f'Candidate markets: {len(inputs)}; complete sample: {len(complete)}; missing-trade exclusions: {len(inputs)-len(complete)}')

Candidate markets: 14; complete sample: 11; missing-trade exclusions: 3


## Results

复算 TOPSIS 并与已存结果逐国比较。排序底稿使用精确表格；不另生成或改动报告图表。

In [2]:
transformed, normalized, lower, upper = itc.transform(complete)
scores, ranks = itc.topsis(normalized, itc.STRATEGY)[:2]
saved = {r['country_code']: r for r in itc.read_csv(root / 'market_ranking_recalculated.csv')}
for i in np.argsort(-scores, kind='stable'):
    r = complete[i]
    assert abs(scores[i]-float(saved[r['country_code']]['strategy_topsis_score'])) < 1e-12
    assert ranks[i] == int(saved[r['country_code']]['rank'])
    print(f"{ranks[i]:2d}  {itc.NAMES[r['country_code']]:8s}  {scores[i]:.6f}")
print('Exact saved-score comparison: PASS')

 1  韩国        0.896282
 2  澳大利亚      0.846239
 3  马来西亚      0.793578
 4  新西兰       0.780820
 5  日本        0.770295
 6  印度尼西亚     0.750149
 7  泰国        0.698047
 8  菲律宾       0.641689
 9  柬埔寨       0.562982
10  新加坡       0.416286
11  文莱        0.000000
Exact saved-score comparison: PASS


### 随机权重复算与原始文件校验

下面复算全部 5,000 组权重排名，并校验所有原始文件指纹。

In [3]:
rng = np.random.default_rng(itc.SEED)
weight_draws = rng.dirichlet(itc.STRATEGY*itc.CONCENTRATION, size=itc.DRAWS)
rank_draws = np.column_stack([itc.topsis(normalized,w)[1] for w in weight_draws])
sensitivity = {r['country_code']: r for r in itc.read_csv(root/'monte_carlo_sensitivity.csv')}
for i,r in enumerate(complete):
    assert abs(float((rank_draws[i]<=3).mean())-float(sensitivity[r['country_code']]['top3_share'])) < 1e-12
assert int((rank_draws<=3).sum()) == 3*itc.DRAWS
raw = Path(RAW_OVERRIDE) if 'RAW_OVERRIDE' in globals() else root.parent/'ITC-data'
files = itc.read_csv(root/'metadata/file_manifest.csv')
for r in files:
    assert hashlib.sha256((raw/r['raw_file']).read_bytes()).hexdigest()==r['sha256']
print(f'Monte Carlo ({itc.DRAWS} draws): PASS; raw file hashes ({len(files)} files): PASS')

Monte Carlo (5000 draws): PASS; raw file hashes (209 files): PASS


## Takeaways

主模型前三不等于企业目标市场已经确定。第三名对权重变化敏感；完整权重方案和留一指标检验见 sensitivity_results.csv 与 model/output/robustness_summary.csv。事实登记和访问时间见 fact_registry.csv，不把重算日期当成数据访问日期。